# Annotate FITS image with transient locator symbols

This notebook will accept as input a dataset spec such as used in file *dataset.json*. 

It then creates a "regions" file for use in SAOimage, with the transients associated with that dataset. The transient input is take from the 'table_candidates_xxx.fits' file, but any other similar file can be used as well.  

In [1]:
import os
import json
from importlib import reload

import numpy as np

from astropy.io import fits
from astropy import units as u
from astropy.table import Table
from astropy.coordinates import SkyCoord
from astropy.wcs import WCS

from regions import PointSkyRegion, CircleSkyRegion, Regions

import settings
from library import get_cutouts, update_dataset
from settings import RESULTS, get_parameters, current_dataset, fname

In [2]:
par = get_parameters(current_dataset)
par

{'nproc': 4,
 'nproc_analysis': 4,
 'use_catalog': True,
 'sextractor_flags': 8,
 'model_prediction': 0.5,
 'elongation': 2.5,
 'annular_bin': 5,
 'flag_rim': 0,
 'max_flux_threshold': 0.1,
 'fwhm_init': 10.0,
 'fit_shape': 33,
 'min_acceptable_flux': 500,
 'min_fwhm': 3.5,
 'max_fwhm': 20.0,
 'qfit_max': 2.5,
 'cfit_max': 0.003,
 'max_fit_flag': 8,
 'neighborhood_cutout_size': 15.0,
 'elongation_limit': 1.8,
 'profile_diff_threshold': 0.08,
 'circularity_threshold': [70],
 'circularity_low_limit': 0.8,
 'tiny_cutout_size': 25,
 'false_positive_threshold': 10.0,
 'fwhm_lim': 30.0,
 'disp_elong_lim': 1.5,
 'display_cutout_size': 2.5,
 'plot_limit': 200,
 'rotate': [False, False],
 'invert_east': [False, False],
 'invert_north': [False, False],
 'table1': 'sources_9099.csv',
 'table2': 'sources_9100.csv',
 'table1_calib': 'sources_calib_9099.csv',
 'table2_calib': 'sources_calib_9100.csv',
 'table_matched': 'table_match_9099_9100.fits',
 'table_psf_matched': 'table_psf_match_9099_9100.fi

In [3]:
table = Table.read(os.path.join(RESULTS, "candidates_best_GS.fits"), format='fits')
m = table['seq'] == 'seq18'
tab = table[m]

# table = Table.read(fname(par['table_candidates']), format='fits')
# tab = table

In [7]:
tab

seq,ut_mid,exptime,es,plate_id_next,ut_mid_next,exptime_next,es_next,source_id,process_id_1,scan_id_1,plate_id_1,archive_id_1,solution_num,annular_bin_1,dist_center_1,dist_edge_1,sextractor_flags_1,model_prediction_1,ra_icrs,dec_icrs,ra_error,dec_error,gal_lon,gal_lat,ecl_lon,ecl_lat,x_sphere,y_sphere,z_sphere,healpix256,healpix1024,nn_dist,zenith_angle,airmass,natmag,natmag_error,bpmag,bpmag_error,rpmag,rpmag_error,natmag_plate,natmag_correction,natmag_residual,phot_range_flags,phot_calib_flags,color_term,cat_natmag,match_radius,gaiaedr3_id,gaiaedr3_gmag,gaiaedr3_bp_rp,gaiaedr3_dist,gaiaedr3_neighbors,timestamp_insert_1,timestamp_update_1,pos,process_id_2,scan_id_2,plate_id_2,archive_id_2,source_num,x_source,y_source,a_source,b_source,theta_source,erra_source,errb_source,errtheta_source,elongation,x_peak,y_peak,flag_usepsf,x_image,y_image,erra_image,errb_image,errtheta_image,x_psf,y_psf,erra_psf,errb_psf,errtheta_psf,mag_auto,magerr_auto,flux_auto,fluxerr_auto,mag_iso,magerr_iso,flux_iso,fluxerr_iso,flux_max,flux_radius,isoarea,sqrt_isoarea,background,sextractor_flags_2,dist_center_2,dist_edge_2,annular_bin_2,flag_rim,flag_negradius,flag_clean,model_prediction_2,timestamp_insert_2,timestamp_update_2,next_plate_id,id,group_id,group_size,local_bkg,x_init,y_init,flux_init,fwhm_init,x_fit,y_fit,flux_fit,fwhm_fit,x_err,y_err,flux_err,fwhm_err,npixfit,qfit,cfit,flags,profile_diff,circularity,area,shape_defect,circle_deviation
bytes5,bytes19,int32,float32,int32,bytes19,int32,float32,int64,int64,int64,int64,int64,int64,int64,float64,float64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,int64,float64,float64,float64,int64,int64,float64,float64,int64,bytes29,bytes29,bytes42,int64,int64,int64,int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,int64,int64,float64,float64,float64,float64,float64,int64,int64,int64,int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,float64,float64,int64,float64,float64,int64,int64,int64,int64,float64,bytes29,bytes29,int64,int64,int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,int64,float64,float64,int64,float64,float64,float64,float64,float64
seq18,1956-04-07 20:30:29,600,68.13718,9100,1956-04-07 21:34:33,600,67.05625,40347160043802,34716,11987,9099,102,0,3,6630.891,5202.0215,0,0.9993973970413208,129.60241652331794,14.247430390355865,0.1463710069656372,0.13667500019073486,211.59778004142308,30.025691174584487,128.2792167671652,-4.08579693983359,-0.6178495669691121,0.7467876986443286,0.24610982456941471,68605,1097693,62.6658935546875,43.74177551269531,1.3822773694992065,--,--,--,--,--,--,0.0,--,--,0,0,--,--,0.0,--,--,--,--,0,2022-06-13 20:48:31.081721+00,2022-06-13 20:48:31.081721+00,"(2.261988886873 , 0.248664570260411)",34716,11987,9099,102,43802,17661.979,13702.13846194218,2.3705697059631348,2.2091734409332275,-42.24782180786133,0.021351251751184464,0.019222527742385864,-48.6621208190918,1.0730572938919067,17662,13704,0,17661.979,13703.701,0.021351251751184464,0.019222527742385864,-48.6621208190918,--,--,--,--,--,9.59018611907959,0.010438338853418827,1458563.75,14019.3212890625,9.740703582763672,0.007494076155126095,1269751.5,8762.076171875,32589.69140625,2.980008125305176,83,9.110433578491211,10145.01953125,0,6630.891,5202.0215,3,0,0,1,0.9993973970413208,2022-06-13 04:59:41.55692+00,2022-06-13 04:59:41.55692+00,9100,276,276,1,0.0,17661.979,13702.13846194218,1059846.68796771,8.0,17660.986418002387,13702.628118801646,1313729.6544719064,5.572172170123504,0.027619994900204503,0.02617440371386158,13178.19041967753,0.043172926191225265,961,0.6340952908919567,-0.0015350569780661302,0,0.023895537403609522,0.8824921179146895,45.0,0.1484375,0.06842521036501593
seq18,1956-04-07 20:30:29,600,68.1371

In [5]:
coords = SkyCoord(ra=tab['ra_icrs']*u.deg, dec=tab['dec_icrs']*u.deg, frame='icrs')

In [6]:
region_list = [CircleSkyRegion(center=c, radius=1.*u.arcmin) for c in coords]

symbols = ['cross', 'diamond']
for i, coord in enumerate(coords):
#     reg = PointSkyRegion(center=coord)
    reg = CircleSkyRegion(center=coord, radius=1.*u.arcmin)
    # Set the symbol type in visual attributes
#     reg.visual['symbol'] = symbols[i % len(symbols)]
    region_list.append(reg)

# Save to a .reg file
regions_obj = Regions(region_list)
regions_obj.write('targets.reg', format='ds9', overwrite=True)